# SEA – Stateful Execution Environment
### End-to-End Quickstart Notebook

This notebook walks you through a full multi-turn conversational analytics session using the **SEA** module.

**Before running**, create a `.env` file in the same directory as this notebook:

```
# .env
GOOGLE_API_KEY=your_google_api_key_here

DATABRICKS_HOSTNAME=dbc-xxxxxxxx-xxxx.cloud.databricks.com
DATABRICKS_HTTP_PATH=/sql/1.0/warehouses/your_warehouse_id
DATABRICKS_TOKEN=dapixxxxxxxxxxxxxxxxxxxxxxxxxxx
DATABRICKS_TABLE_PREFIX=workspace.globomart_db_final

VECTOR_DB_PATH=./data/vector_db_globomart
VECTOR_COLLECTION=globomart_sea
JSON_SUMMARY_DIR=./data/json_summaries
```

---
**Paper**: *SEA: Stateful Execution Environment for Conversational Big Data Analytics* (NeurIPS 2025 Workshop)

## 0 · Install & imports

In [ ]:
# Install the package if not already done
# !pip install -e . --quiet

In [ ]:
import os
import json
import base64
from pathlib import Path
from IPython.display import display, Image, Markdown

# Load .env file
from dotenv import load_dotenv
load_dotenv(override=True)

print("Environment loaded.")
print(f"  GOOGLE_API_KEY     : {'set' if os.getenv('GOOGLE_API_KEY') else 'MISSING'}")
print(f"  DATABRICKS_HOSTNAME: {'set' if os.getenv('DATABRICKS_HOSTNAME') else 'MISSING'}")
print(f"  DATABRICKS_TOKEN   : {'set' if os.getenv('DATABRICKS_TOKEN') else 'MISSING'}")
print(f"  VECTOR_DB_PATH     : {os.getenv('VECTOR_DB_PATH', 'not set')}")
print(f"  JSON_SUMMARY_DIR   : {os.getenv('JSON_SUMMARY_DIR', 'not set')}")

## 1 · Build the SEA instance

The `build_analytics_sea()` factory wires up the 5-node DAG from the paper:

```
SchemaRetriever → QuerySynthesizer → AIAnalytics → ┬─ ChartGenerator
                                                    └─ InsightSummarizer
```

- **Planner** uses Gemini Pro (strategic node-classification)
- **Tools** use Gemini Flash (latency-sensitive code generation)
- State is kept in-memory per session (dual-representation: Main DAG + Twin Summary DAG)

In [ ]:
from sea.workflows.analytics import build_analytics_sea

sea = build_analytics_sea(
    google_api_key        = os.environ["GOOGLE_API_KEY"],
    databricks_hostname   = os.environ["DATABRICKS_HOSTNAME"],
    databricks_http_path  = os.environ["DATABRICKS_HTTP_PATH"],
    databricks_token      = os.environ["DATABRICKS_TOKEN"],
    databricks_table_prefix = os.getenv("DATABRICKS_TABLE_PREFIX", "workspace.globomart_db_final"),
    vector_db_path        = os.getenv("VECTOR_DB_PATH", "./data/vector_db_globomart"),
    vector_collection     = os.getenv("VECTOR_COLLECTION", "globomart_sea"),
    json_summary_dir      = os.getenv("JSON_SUMMARY_DIR", "./data/json_summaries"),
    chart_library         = "matplotlib",
)

print("SEA instance ready.")
print(f"Graph nodes: {list(sea.graph.nodes.keys())}")
print(f"Root node (v0): '{sea.graph.root}'")

### Inspect the DAG structure

In [ ]:
print("Node descriptions (shown to Planner):\n")
for node_id, desc in sea.graph.node_descriptions.items():
    print(f"  [{node_id}]\n    {desc}\n")

print("\nDependency structure:")
print(sea.graph.dependency_description())

print("\nExecution groups from root (parallel groups shown as lists):")
for i, group in enumerate(sea.graph.execution_groups(sea.graph.root)):
    parallel = "(parallel)" if len(group) > 1 else ""
    print(f"  Group {i+1}: {group} {parallel}")

---
## 2 · Helper utilities

In [ ]:
import time

SESSION_ID = "quickstart-session"

def chat(query: str) -> dict:
    """Run one conversational turn and pretty-print the result."""
    print(f"\n{'='*70}")
    print(f"USER ▶  {query}")
    print("="*70)

    t0 = time.perf_counter()
    result = sea.chat(query=query, session_id=SESSION_ID)
    elapsed = time.perf_counter() - t0

    print(f"\n⏱  Latency: {elapsed:.1f}s")
    print(f"🔧 Tool calls: {' → '.join(result['tool_calls'])}")

    dag = result["final_dag_state"]

    # Commentary
    commentary_node = dag.get("insight_summarizer", {})
    if commentary_node.get("status") == "SUCCESS":
        print(f"\n📝 Insights:\n   {commentary_node.get('commentary_snippet', '')}")

    # Charts
    chart_node = dag.get("chart_generator", {})
    if chart_node.get("status") == "SUCCESS":
        charts = chart_node.get("charts_generated", [])
        print(f"\n📊 Charts generated: {charts}")

    return result


def show_charts(result: dict):
    """Decode and display base64 chart images from a result dict."""
    # Charts live in the Main State DAG (not the summary)
    store = sea.session_manager.state_store
    chart_artifact = store.get_artifact(SESSION_ID, "chart_generator")
    charts = chart_artifact.get("charts", [])
    for chart in charts:
        raster = chart.get("raster")
        if raster:
            display(Markdown(f"**{chart.get('viz_type', 'chart')}**"))
            display(Image(data=base64.b64decode(raster), format="png"))


def show_dag_state():
    """Print the current Twin Summary DAG (what the Planner sees)."""
    store = sea.session_manager.state_store
    summary = store.get_summary_dag(SESSION_ID)
    print("\n── Twin Summary DAG (S') ──")
    print(json.dumps(summary, indent=2, default=str))

---
## 3 · Turn 1 – Fresh query (full DAG run)

No prior state → Planner selects `schema_retriever` (v0) → full pipeline executes.

**Expected latency**: ~50-60 s (includes Databricks SQL round-trip).

In [ ]:
result1 = chat("What were the top 5 most profitable products last quarter?")

In [ ]:
show_charts(result1)

In [ ]:
show_dag_state()

---
## 4 · Turn 2 – Visualisation follow-up (state reuse)

The DataFrame is already cached in the Main State DAG.
Planner should enter at `chart_generator` — skipping the expensive Databricks call.

**Expected latency**: ~15-20 s (no SQL, chart gen only).

In [ ]:
result2 = chat("Show the same data as a horizontal bar chart.")

In [ ]:
show_charts(result2)

---
## 5 · Turn 3 – Drill-down analysis (mid-chain re-entry)

The SQL subset is still valid (same tables, same filter scope) but the
analysis needs to change → Planner enters at `ai_analytics`, reusing the
cached DataFrame from `query_synthesizer`.

**Expected latency**: ~20-30 s (no SQL, new pandas analysis + outputs).

In [ ]:
result3 = chat("Break down those top products by product category.")

In [ ]:
show_charts(result3)

---
## 6 · Turn 4 – Topic switch → Purge event

A completely different analytical topic → Planner selects `schema_retriever` (v0) with an
existing non-empty state → **purge event fires**: both DAGs and conversation history are
reset.  The new session starts fresh.

**Expected latency**: ~50-60 s (full pipeline again).

In [ ]:
result4 = chat("What was the total marketing spend by channel last year?")

In [ ]:
show_charts(result4)

---
## 7 · Turn 5 – Follow-up on new topic (state reuse again)

Now that the marketing-budget state is cached, a follow-up should be fast.

In [ ]:
result5 = chat("Compare that budget to total actual spend by channel.")

In [ ]:
show_charts(result5)

---
## 8 · Latency summary

This cell re-runs all 5 turns and records latencies so you can see the
36 % average reduction reported in the paper.

In [ ]:
import time

# Reset session so we measure from scratch
sea.session_manager.clear_session(SESSION_ID)

turns = [
    ("fresh",      "What were the top 5 most profitable products last quarter?"),
    ("stateful",   "Show the same data as a horizontal bar chart."),
    ("stateful",   "Break down those top products by product category."),
    ("fresh",      "What was the total marketing spend by channel last year?"),   # topic switch
    ("stateful",   "Compare that budget to total actual spend by channel."),
]

records = []
for kind, query in turns:
    t0 = time.perf_counter()
    result = sea.chat(query=query, session_id=SESSION_ID)
    elapsed = time.perf_counter() - t0
    records.append({"kind": kind, "query": query[:55] + "…", "latency_s": round(elapsed, 1),
                    "tool_calls": result["tool_calls"]})
    print(f"[{kind:8s}] {elapsed:5.1f}s  {result['tool_calls']}")

fresh_avg    = sum(r["latency_s"] for r in records if r["kind"] == "fresh")    / sum(1 for r in records if r["kind"] == "fresh")
stateful_avg = sum(r["latency_s"] for r in records if r["kind"] == "stateful") / sum(1 for r in records if r["kind"] == "stateful")
reduction    = (fresh_avg - stateful_avg) / fresh_avg * 100

print(f"\nAvg fresh-query latency   : {fresh_avg:.1f}s")
print(f"Avg stateful-query latency: {stateful_avg:.1f}s")
print(f"Latency reduction         : {reduction:.1f}%  (paper reports ~36%)")

---
## 9 · Bring your own graph (custom DAG example)

SEA is not limited to the 5-node analytics workflow. Below is a minimal
example of defining an entirely custom DAG with two mock tools — no
infrastructure required.

In [ ]:
from sea import SEA, SEAGraph, Tool, ToolResult
from sea.core.planner import Planner
from sea.core.schemas import Plan, SessionData, Step


class DataFetcher(Tool):
    name = "data_fetcher"
    description = "Fetches raw records. No dependencies – always the first step."

    def run(self, query, upstream_artifacts, **kwargs):
        print(f"  [DataFetcher] fetching for: {query}")
        records = [{"id": 1, "value": 42}, {"id": 2, "value": 99}]
        return ToolResult(
            artifact={"records": records},
            summary={"status": "SUCCESS", "row_count": len(records)},
        )


class Summariser(Tool):
    name = "summariser"
    description = "Summarises fetched records. Requires data_fetcher output."

    def run(self, query, upstream_artifacts, **kwargs):
        records = upstream_artifacts.get("data_fetcher", {}).get("records", [])
        total = sum(r["value"] for r in records)
        print(f"  [Summariser] total={total}")
        return ToolResult(
            artifact={"total": total, "summary_text": f"Total value is {total}."},
            summary={"status": "SUCCESS", "total": total},
        )


class AlwaysRootPlanner(Planner):
    def plan(self, query, session, graph):
        subgraph = graph.subgraph_from(graph.root)
        return Plan(
            reasoning="Always start from root for demo.",
            enriched_query=query,
            start_node=graph.root,
            execution_plan=[Step(node_id=nid, inputs="{}") for nid in subgraph],
        )


custom_graph = SEAGraph()
custom_graph.add_node("data_fetcher", tool=DataFetcher())
custom_graph.add_node("summariser",   tool=Summariser(), depends_on=["data_fetcher"])

custom_sea = SEA(graph=custom_graph, planner=AlwaysRootPlanner())
out = custom_sea.chat("Summarise the data", session_id="custom-1")
print(f"\nResult summary DAG: {json.dumps(out['final_dag_state'], indent=2)}")

---
## 10 · Optional – Serve over HTTP

Uncomment and run the cell below to expose a `/chat` endpoint (useful for
connecting a frontend or testing with `curl`).

```bash
curl -X POST http://localhost:8080/chat \
     -H 'Content-Type: application/json' \
     -d '{"session_id": "s1", "query": "Top 5 products last quarter"}'
```

In [ ]:
# import uvicorn, threading
# from sea.api.server import create_app
#
# app = create_app(sea)   # use the analytics sea instance from Section 1
#
# def _run():
#     uvicorn.run(app, host="0.0.0.0", port=8080, log_level="warning")
#
# thread = threading.Thread(target=_run, daemon=True)
# thread.start()
# print("Server running at http://localhost:8080  (health: /health)")